# Phase 1 – Form Extraction

This notebook demonstrates the **YOLOv8-based OMR form extraction** pipeline.

## Pipeline Overview

1. Load a raw photo of an OMR sheet (mobile capture).
2. Run **YOLODetector** to find the form bounding box.
3. Apply **FormExtractor** (four-point perspective transform) to produce a
   standardised top-down view.
4. Inspect the **OMRDataset** abstraction for batched loading.

> See `LEGACY_REFERENCE.md` in `Phase_1_Extraction/` for the homography math.

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add Phase 1 src to the Python path so imports resolve correctly.
phase1_src = Path("../Phase_1_Extraction/src").resolve()
if str(phase1_src) not in sys.path:
    sys.path.insert(0, str(phase1_src))

print(f"Phase 1 src path: {phase1_src}")
print(f"Path exists: {phase1_src.exists()}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from yolo_detector import YOLODetector
from form_extractor import FormExtractor
from omr_dataset import OMRDataset

print("Imports successful.")
print(f"YOLODetector: {YOLODetector}")
print(f"FormExtractor: {FormExtractor}")
print(f"OMRDataset: {OMRDataset}")

## 2. YOLODetector – Configuration

`YOLODetector` wraps Ultralytics YOLO and exposes two methods:

| Method | Description |
|---|---|
| `detect(image)` | Returns **all** detections above the confidence threshold |
| `detect_largest(image, label)` | Returns the single detection with the **largest bounding-box area** |

The default confidence threshold is **0.5** – see `LEGACY_REFERENCE.md` for rationale.

In [ ]:
# --- Configuration (update model_path before running) -------------------
MODEL_PATH = "path/to/your/omr_form_detector.pt"   # <-- replace with real path
CONF_THRESHOLD = 0.5
DEVICE = "cpu"

# Instantiate the detector (model is loaded here).
# Uncomment once a real model file is available:
# detector = YOLODetector(MODEL_PATH, conf_threshold=CONF_THRESHOLD, device=DEVICE)
# print(f"Loaded model from {MODEL_PATH}")
print("YOLODetector configuration ready.")
print(f"  model_path    : {MODEL_PATH}")
print(f"  conf_threshold: {CONF_THRESHOLD}")
print(f"  device        : {DEVICE}")

## 3. Loading a Sample Image

In [ ]:
# Replace with a real image path to run end-to-end.
IMAGE_PATH = "path/to/omr_sheet.jpg"   # <-- replace with real path

# Simulate a blank 1080×1920 BGR image for demonstration purposes.
sample_image = np.ones((1920, 1080, 3), dtype=np.uint8) * 240

# Draw a simulated form region (light grey rectangle on near-white background).
cv2.rectangle(sample_image, (100, 200), (980, 1700), (180, 180, 180), -1)
cv2.putText(
    sample_image, "OMR Form Region",
    (250, 950), cv2.FONT_HERSHEY_SIMPLEX, 2, (80, 80, 80), 4
)

plt.figure(figsize=(4, 7))
plt.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
plt.title("Sample Input Image (simulated)")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {sample_image.shape}  (H×W×C, BGR)")

## 4. Detection – `detect` and `detect_largest`

`detect()` returns a list of dicts, each with:
- `label` – class name
- `confidence` – float in [0, 1]
- `bbox` / `bbox_xyxy` – `[x1, y1, x2, y2]` pixel coordinates

In [ ]:
# --- Simulated detection output (mirrors real YOLODetector output schema) ---
simulated_detections = [
    {"label": "form", "confidence": 0.93, "bbox": [100, 200, 980, 1700], "bbox_xyxy": [100, 200, 980, 1700]},
    {"label": "form", "confidence": 0.61, "bbox": [200, 400, 600, 900],  "bbox_xyxy": [200, 400, 600, 900]},
]

print("All detections:")
for i, det in enumerate(simulated_detections):
    x1, y1, x2, y2 = det["bbox"]
    area = (x2 - x1) * (y2 - y1)
    print(f"  [{i}] label={det['label']}  conf={det['confidence']:.2f}  area={area:,} px²")

# Select the largest.
largest = max(simulated_detections, key=lambda d: (d["bbox"][2]-d["bbox"][0])*(d["bbox"][3]-d["bbox"][1]))
print(f"\nLargest detection: {largest}")

## 5. FormExtractor – Perspective Transform

`FormExtractor._four_point_transform` sorts the four bounding-box corners into
*(top-left, top-right, bottom-right, bottom-left)* order, then calls
`cv2.getPerspectiveTransform` to compute the 3×3 homography **H** and
`cv2.warpPerspective` to apply it.

In [ ]:
# Demonstrate the four-point transform directly (no model needed).
# We simulate a slightly skewed quadrilateral.

class _MockDetector:
    """Minimal mock that returns a fixed detection for demonstration."""
    def detect_largest(self, image, label=None):
        return {"label": "form", "confidence": 0.93, "bbox": [100.0, 200.0, 980.0, 1700.0], "bbox_xyxy": [100.0, 200.0, 980.0, 1700.0]}

extractor = FormExtractor(_MockDetector(), target_size=(400, 500))

extracted = extractor.extract(sample_image)

if extracted is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original ({sample_image.shape[1]}×{sample_image.shape[0]})")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(extracted, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Extracted ({extracted.shape[1]}×{extracted.shape[0]})")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Extracted form shape: {extracted.shape}")
else:
    print("No form detected.")

## 6. OMRDataset – Batched Loading

`OMRDataset` is a standard `torch.utils.data.Dataset` that:
- Loads images as BGR NumPy arrays via OpenCV.
- Optionally stores labels (int, dict, or any type).
- Supports a `transform` callable for on-the-fly augmentation.

In [ ]:
import torch
from torch.utils.data import DataLoader

# In practice, provide real image paths.
# Here we demonstrate the API with a placeholder list.
image_paths = ["sheet_001.jpg", "sheet_002.jpg", "sheet_003.jpg"]
labels = [0, 1, 0]   # example integer labels

dataset = OMRDataset(image_paths=image_paths, labels=labels)
print(f"Dataset length: {len(dataset)}")
print(f"Image paths  : {[str(p) for p in dataset.image_paths]}")
print(f"Labels       : {dataset.labels}")

# DataLoader usage (images must exist on disk for __getitem__ to succeed).
# loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0)
print("\nOMRDataset API demonstration complete.")

## Summary

| Component | Role |
|---|---|
| `YOLODetector` | Finds the form bounding box using a trained YOLOv8 model |
| `FormExtractor` | Applies four-point perspective warp → standardised 800×1000 image |
| `OMRDataset` | PyTorch Dataset for batched loading with optional transforms |

The extracted form is fed into **Phase 2 – Preprocessing** for CLAHE normalisation.